# Conditional Flow Matching — Cell Morphology with FiLM PCA Conditioning
Learns the conditional distribution `p(y | c)` where:
- `y` = 56 morphological features per cell  
- `c` = 50-dimensional PCA representation of gene expression from `adata.obsm["X_pca"]`  
- FiLM layers modulate the morphology vector field with `c` and time  

Uses OT-Conditional Flow Matching from `torchcfm`.

## 1 · Setup

In [4]:
# Install extra deps not bundled in the Kaggle base image
%pip install -q torchcfm torchdiffeq anndata

/Users/hannesneumann/Repositories/conditional-flow-matching-seminar/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

import contextlib
import json
import math
import os
import time
from dataclasses import dataclass, asdict

import anndata
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchdiffeq
import torch.nn.functional as F
import wandb
from scipy.stats import wasserstein_distance
from torch.utils.data import DataLoader, TensorDataset
from torchcfm.conditional_flow_matching import ExactOptimalTransportConditionalFlowMatcher

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

## 2 · Config

In [ ]:
@dataclass
class Config:
    data_path: str = "/kaggle/input/datasets/cakeflight/morphology-dataset/whole_dataset_train_val.h5ad"
    output_dir: str = "/kaggle/working/outputs_film_pca"

    # Architecture
    y_dim: int = 56
    c_dim: int = 50
    hidden_dim: int = 512
    n_res_blocks: int = 6
    time_emb_dim: int = 128

    # Training
    seed: int = 42
    batch_size: int = 1024
    n_steps: int = 1_000
    lr: float = 2e-4
    weight_decay: float = 1e-4
    warmup_steps: int = 1000
    ema_decay: float = 0.9999
    sigma: float = 0.0
    leiden_key: str = "leiden"
    val_fraction: float = 0.1

    # Logging / validation
    log_every: int = 100
    val_every: int = 1000

    # Eval
    n_eval_samples: int = 4096
    n_eval_samples_per_cluster: int = 2048
    ode_rtol: float = 1e-5
    ode_atol: float = 1e-5


cfg = Config(
    # ↓ override only what differs from defaults
    # n_steps=500,  # smoke test
)
os.makedirs(cfg.output_dir, exist_ok=True)
print(asdict(cfg))

In [ ]:
# W&B login — reads WANDB_API_KEY from Kaggle Secrets (Add-ons → Secrets in the notebook editor).
# Falls back to anonymous mode so the notebook still runs without a secret set.
try:
    from kaggle_secrets import UserSecretsClient
    _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    wandb.login(key=_key)
    print("Logged in to W&B via Kaggle secret.")
except Exception as _e:
    wandb.login(anonymous="allow")
    print(f"W&B anonymous mode (Kaggle secret not found: {_e})")

## 3 · Model

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """(B,) -> (B, dim)"""
    if t.dim() == 0:
        t = t.unsqueeze(0)
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000)
        * torch.arange(half, dtype=torch.float32, device=t.device)
        / max(half - 1, 1)
    )
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, dim: int, cond_dim: int) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.linear1 = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.linear2 = nn.Linear(dim, dim)
        self.film = nn.Linear(cond_dim, 2 * dim)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        gamma, beta = self.film(cond).chunk(2, dim=-1)
        h = self.linear1(F.silu(self.norm1(x)))
        h = (1 + gamma) * h + beta
        h = self.linear2(F.silu(self.norm2(h)))
        return x + h


class MorphologyVectorField(nn.Module):
    def __init__(
        self,
        y_dim: int,
        c_dim: int,
        hidden_dim: int,
        n_res_blocks: int,
        time_emb_dim: int,
    ) -> None:
        super().__init__()
        self.time_emb_dim = time_emb_dim

        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )

        cond_dim = c_dim + time_emb_dim
        self.input_proj = nn.Linear(y_dim, hidden_dim)
        self.res_blocks = nn.ModuleList(
            [ResBlock(hidden_dim, cond_dim) for _ in range(n_res_blocks)]
        )
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, y_dim)

    def forward(
        self, y_t: torch.Tensor, t: torch.Tensor, x_pca: torch.Tensor
    ) -> torch.Tensor:
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_emb_dim))
        cond = torch.cat([x_pca, t_emb], dim=-1)

        h = self.input_proj(y_t)
        for block in self.res_blocks:
            h = block(h, cond)
        return self.output_head(self.output_norm(h))

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9999) -> None:
        self.decay = decay
        self.shadow: dict[str, torch.Tensor] = {
            k: v.clone().detach() for k, v in model.state_dict().items()
        }

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    def apply_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow)


## 4 · Data

In [ ]:
_DROP_COLUMNS = {
    "Center_X", "Center_Y",
    "BoundingBoxMinimum_X", "BoundingBoxMaximum_X",
    "BoundingBoxMinimum_Y", "BoundingBoxMaximum_Y",
    "Orientation",
    "NormalizedMoment_1_0", "NormalizedMoment_0_0", "NormalizedMoment_0_1"
}


def get_morphology_columns(obs_columns: list[str]) -> list[str]:
    cols = [c for c in obs_columns if c[0].isupper()]
    cols = [c for c in cols if c not in _DROP_COLUMNS]
    cols = [c for c in cols if not c.startswith("SpatialMoment_")]
    assert len(cols) == cfg.y_dim, (
        f"Expected {cfg.y_dim} morphological features, got {len(cols)}.\nColumns: {cols}"
    )
    return cols


def split_train_val_by_leiden_clusters(
    leiden: np.ndarray,
    val_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, dict[str, list[str]]]:
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    labels, counts = np.unique(leiden.astype(str), return_counts=True)
    rng = np.random.default_rng(seed)
    tie_break = rng.permutation(len(labels))
    order = np.lexsort((tie_break, -counts))

    n = len(leiden)
    target_sizes = {
        "train": n * (1.0 - val_fraction),
        "val": n * val_fraction,
    }
    split_sizes = {"train": 0, "val": 0}
    split_clusters: dict[str, list[str]] = {"train": [], "val": []}

    for i in order:
        label = str(labels[i])
        count = int(counts[i])
        split = max(
            split_sizes,
            key=lambda name: target_sizes[name] - split_sizes[name],
        )
        split_clusters[split].append(label)
        split_sizes[split] += count

    leiden_str = leiden.astype(str)
    train_idx = np.flatnonzero(np.isin(leiden_str, split_clusters["train"]))
    val_idx = np.flatnonzero(np.isin(leiden_str, split_clusters["val"]))
    return train_idx, val_idx, split_clusters


def load_data(cfg: Config):
    print(f"Loading {cfg.data_path} ...")
    adata = anndata.read_h5ad(cfg.data_path)

    c = torch.tensor(np.array(adata.obsm["X_pca"]), dtype=torch.float32)
    assert c.shape[1] == cfg.c_dim, f"Expected c.shape[1]=={cfg.c_dim}, got {c.shape[1]}"

    morph_cols = get_morphology_columns(list(adata.obs.columns))
    y = torch.tensor(adata.obs[morph_cols].values.astype(np.float32), dtype=torch.float32)
    assert y.shape[1] == cfg.y_dim, f"Expected y.shape[1]=={cfg.y_dim}, got {y.shape[1]}"
    if cfg.leiden_key not in adata.obs:
        raise KeyError(f"Missing Leiden column {cfg.leiden_key!r} in adata.obs")
    leiden = adata.obs[cfg.leiden_key].astype(str).to_numpy()

    print(f"c={tuple(c.shape)}, y={tuple(y.shape)}")

    # Impute NaN morphological features with per-feature median
    nan_mask = torch.isnan(y)
    if nan_mask.any():
        n_nan_cells = nan_mask.any(dim=1).sum().item()
        n_nan_vals  = nan_mask.sum().item()
        print(f"Imputing {n_nan_vals} NaN values across {n_nan_cells} cells ({n_nan_cells/len(y)*100:.1f}%) with per-feature median")
        for j in range(y.shape[1]):
            col = y[:, j]
            median = col[~torch.isnan(col)].median()
            y[:, j] = torch.where(torch.isnan(col), median, col)

    # Impute inf values with per-feature median of finite values
    inf_mask = torch.isinf(y)
    if inf_mask.any():
        n_inf_cells = inf_mask.any(dim=1).sum().item()
        n_inf_vals  = inf_mask.sum().item()
        print(f"Imputing {n_inf_vals} inf values across {n_inf_cells} cells ({n_inf_cells/len(y)*100:.1f}%) with per-feature median")
        for j in range(y.shape[1]):
            col = y[:, j]
            finite_vals = col[torch.isfinite(col)]
            if finite_vals.numel() > 0:
                median = finite_vals.median()
                y[:, j] = torch.where(torch.isinf(col), median, col)

    assert not torch.isnan(y).any() and not torch.isnan(c).any(), "NaN values remain after imputation"
    assert torch.isfinite(y).all() and torch.isfinite(c).all(), "Non-finite values remain after imputation"

    # Raw CellProfiler features span many orders of magnitude and include negative values
    # (Hu moments, central moments). Signed log1p compresses the dynamic range while
    # preserving sign, making z-score standardization well-behaved.
    y = torch.sign(y) * torch.log1p(torch.abs(y))
    print(
        f"After signed log1p: min={y.min().item():.3f}, max={y.max().item():.3f}, "
        f"any NaN={torch.isnan(y).any().item()}, any inf={torch.isinf(y).any().item()}"
    )

    train_idx, val_idx, split_clusters = split_train_val_by_leiden_clusters(
        leiden=leiden,
        val_fraction=cfg.val_fraction,
        seed=cfg.seed,
    )
    print("Leiden cluster split:")
    for name, idx in (("train", train_idx), ("val", val_idx)):
        clusters = split_clusters[name]
        print(
            f"  {name:5s}: {len(idx):7,d} cells | "
            f"{len(clusters):2d} clusters | {', '.join(clusters)}"
        )

    c_train, c_val = c[train_idx], c[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    leiden_train, leiden_val = leiden[train_idx], leiden[val_idx]

    # z-score using train statistics only
    y_mean = y_train.mean(0)
    y_std  = y_train.std(0).clamp(min=1e-8)
    y_train = (y_train - y_mean) / y_std
    y_val   = (y_val   - y_mean) / y_std

    abs_max = y_train.abs().max(dim=0).values
    top = torch.topk(abs_max, k=5)
    print("Top-5 features by |max| after standardization:")
    for i, idx in enumerate(top.indices.tolist()):
        print(f"  {morph_cols[idx]:30s} |max|={top.values[i].item():.2e}")

    return (
        c_train, y_train, leiden_train,
        c_val, y_val, leiden_val,
        y_mean, y_std, morph_cols, split_clusters,
    )

In [ ]:
(
    c_train, y_train, leiden_train,
    c_val, y_val, leiden_val,
    y_mean, y_std,
    morph_cols, split_clusters,
) = load_data(cfg)

torch.save(
    {"mean": y_mean, "std": y_std, "columns": morph_cols},
    os.path.join(cfg.output_dir, "y_stats.pt"),
)
with open(os.path.join(cfg.output_dir, "split_clusters.json"), "w") as f:
    json.dump(split_clusters, f, indent=2)
print(f"Train: {len(c_train):,}  |  Val: {len(c_val):,}")

## 5 · Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

num_workers = 2 if device.type == "cuda" else 0
pin = device.type == "cuda"

train_loader = DataLoader(
    TensorDataset(c_train, y_train),
    batch_size=cfg.batch_size, shuffle=True,
    drop_last=True, num_workers=num_workers, pin_memory=pin,
)
val_loader = DataLoader(
    TensorDataset(c_val, y_val),
    batch_size=cfg.batch_size * 2, shuffle=False,
    drop_last=False, num_workers=num_workers, pin_memory=pin,
)

model = MorphologyVectorField(
    y_dim=cfg.y_dim, c_dim=cfg.c_dim,
    hidden_dim=cfg.hidden_dim, n_res_blocks=cfg.n_res_blocks,
    time_emb_dim=cfg.time_emb_dim,
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

ema = EMA(model, decay=cfg.ema_decay)
FM  = ExactOptimalTransportConditionalFlowMatcher(sigma=cfg.sigma)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

def _lr_lambda(step: int) -> float:
    return float(step) / max(cfg.warmup_steps, 1) if step < cfg.warmup_steps else 1.0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

use_amp  = device.type == "cuda"
scaler   = torch.cuda.amp.GradScaler(enabled=use_amp)
amp_ctx  = (
    torch.autocast(device_type="cuda", dtype=torch.float16)
    if use_amp else contextlib.nullcontext()
)

run = wandb.init(
    project="cfm-morphology",
    # name weglassen → wandb generiert "adjective-noun-N"
    group=f"cfm_h{cfg.hidden_dim}_r{cfg.n_res_blocks}",
    tags=[f"hidden{cfg.hidden_dim}", f"res{cfg.n_res_blocks}", f"seed{cfg.seed}"],
    config=asdict(cfg),
    resume="allow",
)
print(f"W&B run: {run.url}")

In [ ]:
@torch.no_grad()
def val_loss() -> float:
    model.eval()
    total, count = 0.0, 0
    for c_b, y1_b in val_loader:
        c_b  = c_b.to(device, non_blocking=True)
        y1_b = y1_b.to(device, non_blocking=True)
        with amp_ctx:
            t, yt, ut = FM.sample_location_and_conditional_flow(torch.randn_like(y1_b), y1_b)
            loss = nn.functional.mse_loss(model(yt, t, c_b), ut)
        total += loss.item() * len(c_b)
        count += len(c_b)
    model.train()
    return total / count


train_losses, train_steps = [], []
val_losses,   val_steps   = [], []

running_loss, running_count = 0.0, 0
step = 0
t0   = time.time()
data_iter = iter(train_loader)

model.train()
while step < cfg.n_steps:
    try:
        c_b, y1_b = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        c_b, y1_b = next(data_iter)

    c_b  = c_b.to(device, non_blocking=True)
    y1_b = y1_b.to(device, non_blocking=True)

    optimizer.zero_grad()
    with amp_ctx:
        t, yt, ut = FM.sample_location_and_conditional_flow(torch.randn_like(y1_b), y1_b)
        loss = nn.functional.mse_loss(model(yt, t, c_b), ut)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    ema.update(model)

    running_loss  += loss.item()
    running_count += 1
    step += 1

    if step % cfg.log_every == 0:
        avg   = running_loss / running_count
        lr_now = scheduler.get_last_lr()[0]
        ms    = (time.time() - t0) / cfg.log_every * 1000
        print(f"step {step:6d}/{cfg.n_steps}  loss={avg:.5f}  lr={lr_now:.2e}  {ms:.1f}ms/step")
        train_losses.append(avg)
        train_steps.append(step)
        wandb.log({"train/loss": avg, "train/lr": lr_now, "train/ms_per_step": ms}, step=step)
        running_loss = running_count = 0
        t0 = time.time()

    if step % cfg.val_every == 0:
        vl = val_loss()
        val_losses.append(vl)
        val_steps.append(step)
        print(f"  → val loss={vl:.5f}")
        wandb.log({"val/loss": vl}, step=step)

print("Training complete.")

In [ ]:
# Save EMA model weights
ema_model = MorphologyVectorField(
    y_dim=cfg.y_dim, c_dim=cfg.c_dim,
    hidden_dim=cfg.hidden_dim, n_res_blocks=cfg.n_res_blocks,
    time_emb_dim=cfg.time_emb_dim,
).to(device)
ema.apply_to(ema_model)
torch.save(ema_model.state_dict(), os.path.join(cfg.output_dir, "model_ema.pt"))

# Save config
with open(os.path.join(cfg.output_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

print(f"Saved model + config to {cfg.output_dir}/")

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps, train_losses, label="train", alpha=0.8)
if val_losses:
    ax.plot(val_steps, val_losses, label="val", marker="o", markersize=3)
ax.set_xlabel("step")
ax.set_ylabel("MSE loss")
ax.set_title("CFM Training Loss")
ax.legend()
plt.tight_layout()
loss_curves_path = os.path.join(cfg.output_dir, "loss_curves.png")
plt.savefig(loss_curves_path, dpi=100)
wandb.log({"charts/loss_curves": wandb.Image(loss_curves_path)})
plt.show()

## 6 · Evaluation

In [ ]:
ema_model.eval()

@torch.no_grad()
def sample_morphology(c_eval: torch.Tensor, y_real_norm: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    n = len(c_eval)
    c_eval = c_eval.to(device)
    print(f"Sampling {n} validation cells with dopri5 ODE solver ...")
    y0 = torch.randn(n, cfg.y_dim, device=device)

    def ode_fn(t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return ema_model(y, t.expand(y.shape[0]), c_eval)

    traj = torchdiffeq.odeint(
        ode_fn, y0,
        torch.tensor([0.0, 1.0], device=device),
        method="dopri5",
        rtol=cfg.ode_rtol,
        atol=cfg.ode_atol,
        options={"dtype": torch.float32},
    )

    # Inverse z-score, then inverse signed log1p -> original CellProfiler scale
    y_real_log = y_real_norm * y_std + y_mean
    y_gen_log  = traj[-1].cpu() * y_std + y_mean
    y_real = (torch.sign(y_real_log) * torch.expm1(torch.abs(y_real_log))).numpy()
    y_gen  = (torch.sign(y_gen_log)  * torch.expm1(torch.abs(y_gen_log))).numpy()
    return y_real, y_gen


def mean_w1(y_real_arr: np.ndarray, y_gen_arr: np.ndarray) -> float:
    return float(np.mean([
        wasserstein_distance(y_real_arr[:, i], y_gen_arr[:, i])
        for i in range(y_real_arr.shape[1])
    ]))


rng = np.random.default_rng(cfg.seed)
leiden_val_str = np.asarray(leiden_val).astype(str)
validation_clusters = np.unique(leiden_val_str)
print(f"Validation evaluation over Leiden clusters: {', '.join(validation_clusters)}")

y_real_parts, y_gen_parts = [], []
cluster_metrics = []
for cluster in validation_clusters:
    cluster_idx = np.flatnonzero(leiden_val_str == cluster)
    n_cluster = min(cfg.n_eval_samples_per_cluster, len(cluster_idx))
    chosen_idx = rng.choice(cluster_idx, size=n_cluster, replace=False)
    y_real_cluster, y_gen_cluster = sample_morphology(c_val[chosen_idx], y_val[chosen_idx])
    cluster_w1 = mean_w1(y_real_cluster, y_gen_cluster)
    cluster_metrics.append({
        "leiden": str(cluster),
        "n_available": int(len(cluster_idx)),
        "n_evaluated": int(n_cluster),
        "mean_w1": cluster_w1,
    })
    y_real_parts.append(y_real_cluster)
    y_gen_parts.append(y_gen_cluster)
    print(f"  leiden {cluster:>4s}: {n_cluster:5,d}/{len(cluster_idx):5,d} cells | mean W1={cluster_w1:.4f}")

y_real = np.concatenate(y_real_parts, axis=0)
y_gen = np.concatenate(y_gen_parts, axis=0)
validation_mean_w1 = mean_w1(y_real, y_gen)
metrics_path = os.path.join(cfg.output_dir, "validation_cluster_metrics.json")
with open(metrics_path, "w") as f:
    json.dump({"split": "validation", "clusters": cluster_metrics, "mean_w1": validation_mean_w1}, f, indent=2)
print(f"Validation combined: y_real={y_real.shape}, y_gen={y_gen.shape}, mean W1={validation_mean_w1:.4f}")
print(f"Saved validation cluster metrics to {metrics_path}")


In [ ]:
# Per-feature marginal stats + Wasserstein-1
w1_list = []
header = f"{'Feature':<42} {'real_mean':>10} {'gen_mean':>10} {'real_std':>10} {'gen_std':>10} {'W1':>10}"
print(header)
print("-" * len(header))
for i, col in enumerate(morph_cols):
    rm, gm = y_real[:, i].mean(), y_gen[:, i].mean()
    rs, gs = y_real[:, i].std(),  y_gen[:, i].std()
    w1 = wasserstein_distance(y_real[:, i], y_gen[:, i])
    w1_list.append(w1)
    print(f"{col:<42} {rm:>10.4f} {gm:>10.4f} {rs:>10.4f} {gs:>10.4f} {w1:>10.4f}")

mean_w1 = np.mean(w1_list)
print(f"\nMean W1: {mean_w1:.4f}")

wandb.log({
    "validation/mean_w1": mean_w1,
    **{f"validation/w1/{col}": w1 for col, w1 in zip(morph_cols, w1_list)},
})

In [ ]:
# Per-feature histogram grid (56 subplots)
ncols_g = 8
nrows_g = math.ceil(len(morph_cols) / ncols_g)
fig, axes = plt.subplots(nrows_g, ncols_g, figsize=(ncols_g * 3, nrows_g * 2.5))
axes = axes.flatten()

for i, col in enumerate(morph_cols):
    ax = axes[i]
    ax.hist(y_real[:, i], bins=50, alpha=0.5, density=True, color="steelblue", label="real")
    ax.hist(y_gen[:, i],  bins=50, alpha=0.5, density=True, color="tomato",    label="gen")
    ax.set_title(col, fontsize=6)
    ax.set_yticks([])
    ax.tick_params(labelsize=5)

for j in range(len(morph_cols), len(axes)):
    axes[j].set_visible(False)

axes[0].legend(fontsize=7)
fig.suptitle("Marginals: real (blue) vs generated (red)", fontsize=11)
plt.tight_layout()
marginals_path = os.path.join(cfg.output_dir, "validation_marginals.png")
plt.savefig(marginals_path, dpi=100, bbox_inches="tight")
wandb.log({"charts/validation_marginals": wandb.Image(marginals_path)})
plt.show()

In [ ]:
# Pearson correlation matrices
corr_r = np.corrcoef(y_real.T)
corr_g = np.corrcoef(y_gen.T)
diff   = np.abs(corr_r - corr_g)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(corr_r, vmin=-1, vmax=1, cmap="RdBu_r")
axes[0].set_title("Real correlations")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(corr_g, vmin=-1, vmax=1, cmap="RdBu_r")
axes[1].set_title("Generated correlations")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(diff, vmin=0, vmax=1, cmap="hot_r")
axes[2].set_title("Absolute difference")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
correlations_path = os.path.join(cfg.output_dir, "validation_correlations.png")
plt.savefig(correlations_path, dpi=100, bbox_inches="tight")
wandb.log({
    "charts/validation_correlations": wandb.Image(correlations_path),
    "validation/mean_corr_diff": float(diff.mean()),
})
plt.show()

wandb.finish()